# Continuous Window YOLO Dog Count Estimator

Goal:

`raw_videos.json → sample continuous short windows → YOLO dog boxes → max visible dog count → JSON`

This notebook is meant to replace the messy merged notebook. It keeps the original metadata-counting goal, but borrows the useful teammate idea: process continuous frames and save debug box images.

Main output field:

`num_dogs_visible_estimated`

Main rule:

`final_count = max(all_frame_counts)`

That matches the target: maximum number of visible dogs at any point in the video.

In [1]:
import os
import json
import time
import math
from pathlib import Path
from collections import Counter

import cv2
import numpy as np
from ultralytics import YOLO

In [ ]:
# =========================
# CONFIG
# =========================

# All runtime paths flow from BASE_DIR.
# Set BASE_DIR to the server location when running on the server;
# this notebook is meant to be edited locally and executed on the server
# without path changes.
BASE_DIR = "/supernova/data/home/aryah"

METADATA_PATH = f"{BASE_DIR}/raw_videos.json"
OUTPUT_DIR = BASE_DIR
MODEL_PATH = f"{BASE_DIR}/yolo26n.pt"

# Optional ID list. Set to None if you want to select directly from raw_videos.json.
ID_LIST_PATH = None

FAILED_ZERO_VIDEO_IDS = [
    2, 3, 6, 8, 130, 149, 166, 168, 178, 184,
    229, 313, 314, 319, 320, 360, 364, 421,
    432, 487, 556
]

OUTPUT_JSON = f"{OUTPUT_DIR}/yolo26_continuous_failed_zero_retest_conf025.json"

DEBUG_DIR = f"{OUTPUT_DIR}/yolo26_debug_failed_zero_retest"

# Original sampling plan
COVERAGE = 0.40
WINDOW_SIZE = 2.0

# New teammate-inspired change:
# process continuous frames inside each window, not just sparse random/linspace frames.
FRAME_STRIDE = 3

# YOLO settings
YOLO_INTERNAL_CONF = 0.05   # let YOLO return weak possible boxes
CONF_THRESHOLD = 0.25       # manually keep dog boxes above this confidence
YOLO_NMS_IOU = 0.45         # IMPORTANT: do not use tracker IoU like 0.08 here
IMGSZ = 1280

DEVICE = 0

# Testing settings
MAX_VIDEOS = 5
MAX_DURATION = 100
MAX_ANCHORS = None

SAVE_DEBUG_IMAGES = True
MAX_DEBUG_IMAGES_PER_VIDEO = 3

In [3]:
# =========================
# LOAD MODEL
# =========================

model = YOLO(MODEL_PATH)
print(model.names)

{0: 'person', 1: 'bicycle', 2: 'car', 3: 'motorcycle', 4: 'airplane', 5: 'bus', 6: 'train', 7: 'truck', 8: 'boat', 9: 'traffic light', 10: 'fire hydrant', 11: 'stop sign', 12: 'parking meter', 13: 'bench', 14: 'bird', 15: 'cat', 16: 'dog', 17: 'horse', 18: 'sheep', 19: 'cow', 20: 'elephant', 21: 'bear', 22: 'zebra', 23: 'giraffe', 24: 'backpack', 25: 'umbrella', 26: 'handbag', 27: 'tie', 28: 'suitcase', 29: 'frisbee', 30: 'skis', 31: 'snowboard', 32: 'sports ball', 33: 'kite', 34: 'baseball bat', 35: 'baseball glove', 36: 'skateboard', 37: 'surfboard', 38: 'tennis racket', 39: 'bottle', 40: 'wine glass', 41: 'cup', 42: 'fork', 43: 'knife', 44: 'spoon', 45: 'bowl', 46: 'banana', 47: 'apple', 48: 'sandwich', 49: 'orange', 50: 'broccoli', 51: 'carrot', 52: 'hot dog', 53: 'pizza', 54: 'donut', 55: 'cake', 56: 'chair', 57: 'couch', 58: 'potted plant', 59: 'bed', 60: 'dining table', 61: 'toilet', 62: 'tv', 63: 'laptop', 64: 'mouse', 65: 'remote', 66: 'keyboard', 67: 'cell phone', 68: 'microw

In [4]:
# =========================
# PATH / FILE HELPERS
# =========================

def fix_video_path(path):
    """
    Your metadata paths may point to /data/shared/private,
    but on supernova the files are under /redgiant/data/shared/private.
    """
    if path is None:
        return None

    return path.replace(
        "/data/shared/private",
        "/redgiant/data/shared/private"
    )


def load_video_ids(path):
    """
    Load optional list of video IDs from json/txt.
    """
    if path is None:
        return None

    with open(path, "r") as f:
        if path.endswith(".json"):
            ids = json.load(f)
        else:
            ids = [line.strip() for line in f if line.strip()]

    return set(int(x) for x in ids)

In [6]:
# =========================
# ANCHOR SELECTION
# =========================

def get_anchor_times(duration, target_coverage=0.40, window_size=2.0):
    """
    Keep the original plan:
    - short videos: important fixed percentages
    - medium videos: spread windows across video, force early/mid/late anchors
    - long videos: spread windows and add required anchor times
    """

    if duration <= 30:
        anchor_times = [
            0.10 * duration,
            0.25 * duration,
            0.50 * duration,
            0.75 * duration,
            0.90 * duration,
        ]
        video_type = "short"

    elif duration <= 120:
        num_anchors = int(np.ceil((target_coverage * duration) / window_size))

        anchor_times = np.linspace(
            window_size / 2,
            duration - window_size / 2,
            num_anchors
        ).tolist()

        required = [
            5,
            15,
            30,
            0.10 * duration,
            0.50 * duration,
            0.90 * duration,
        ]

        required = [t for t in required if 0 < t < duration]
        anchor_times.extend(required)

        anchor_times = sorted(set(round(t, 2) for t in anchor_times))
        video_type = "medium"

    else:
        num_anchors = int(np.ceil((target_coverage * duration) / window_size))

        anchor_times = np.linspace(
            window_size / 2,
            duration - window_size / 2,
            num_anchors
        ).tolist()

        required = [
            5,
            15,
            30,
            0.10 * duration,
            0.25 * duration,
            0.50 * duration,
            0.75 * duration,
            0.90 * duration,
        ]

        required = [t for t in required if 0 < t < duration]
        anchor_times.extend(required)

        anchor_times = sorted(set(round(t, 2) for t in anchor_times))
        video_type = "long"

    return video_type, anchor_times

In [7]:
# =========================
# YOLO DOG DETECTOR
# =========================

def detect_dog_boxes(frame, model):
    """
    Run YOLO and return only dog boxes above CONF_THRESHOLD.

    Important:
    - YOLO_INTERNAL_CONF is low so possible dogs are not removed too early.
    - CONF_THRESHOLD is the actual dog confidence filter.
    - YOLO_NMS_IOU should be around 0.45, not 0.08.
    """

    results = model.predict(
        source=frame,
        verbose=False,
        device=DEVICE,
        imgsz=IMGSZ,
        conf=YOLO_INTERNAL_CONF,
        iou=YOLO_NMS_IOU
    )

    r = results[0]
    dog_boxes = []

    if r.boxes is None:
        return dog_boxes

    boxes = r.boxes.xyxy.cpu().numpy()
    cls_ids = r.boxes.cls.cpu().numpy()
    confs = r.boxes.conf.cpu().numpy()

    for box, cls_id, conf in zip(boxes, cls_ids, confs):
        label = model.names[int(cls_id)]

        if label == "dog" and float(conf) >= CONF_THRESHOLD:
            x1, y1, x2, y2 = box

            dog_boxes.append({
                "box": [int(x1), int(y1), int(x2), int(y2)],
                "conf": float(conf),
                "label": label
            })

    return dog_boxes

In [8]:
# =========================
# DRAW DEBUG BOXES
# =========================

def draw_dog_boxes(frame, dog_boxes, video_id, frame_idx, t):
    """
    Draw dog boxes for saved debug images.
    This is the useful visual-debugging part inspired by your teammate's notebook.
    """

    out = frame.copy()

    for item in dog_boxes:
        x1, y1, x2, y2 = item["box"]
        conf = item["conf"]

        cv2.rectangle(out, (x1, y1), (x2, y2), (255, 0, 0), 3)

        text = f"dog {conf:.2f}"
        cv2.rectangle(out, (x1, max(0, y1 - 30)), (x1 + 130, y1), (255, 0, 0), -1)

        cv2.putText(
            out,
            text,
            (x1 + 4, max(20, y1 - 8)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (255, 255, 255),
            2
        )

    header = f"video_id={video_id} frame={frame_idx} time={t:.2f}s dogs={len(dog_boxes)}"

    cv2.putText(
        out,
        header,
        (20, 40),
        cv2.FONT_HERSHEY_SIMPLEX,
        1.0,
        (0, 255, 255),
        2
    )

    return out

In [9]:
# =========================
# OPTIONAL DEBUG STABILITY
# =========================

def stable_count_debug(counts, min_fraction=0.30, min_frames=3):
    """
    Keep this only for debugging.
    Do NOT use it as the final count right now.
    """

    if not counts:
        return 0

    threshold = max(min_frames, math.ceil(min_fraction * len(counts)))
    freq = Counter(counts)

    for count in sorted(freq.keys(), reverse=True):
        if freq[count] >= threshold:
            return count

    return 0

In [10]:
# =========================
# MAIN ESTIMATOR FOR ONE VIDEO
# =========================

def estimate_num_dogs_continuous_windows(video_path, video_id=None, duration=None):
    """
    Estimate num_dogs_visible_estimated using continuous frames inside anchor windows.

    Main rule:
        final_count = max dog boxes seen in any processed frame

    This matches:
        maximum number of visible dogs at any point in the video.
    """

    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():
        return -1, {
            "error": "could_not_open_video",
            "video_path": video_path
        }

    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if duration is None or duration <= 0:
        duration = total_frames / fps if fps and fps > 0 else 0

    if duration <= 0 or fps <= 0 or total_frames <= 0:
        cap.release()
        return -1, {
            "error": "bad_duration_or_fps",
            "video_path": video_path,
            "fps": fps,
            "total_frames": total_frames,
            "duration": duration
        }

    video_type, anchors = get_anchor_times(
        duration=duration,
        target_coverage=COVERAGE,
        window_size=WINDOW_SIZE
    )

    if MAX_ANCHORS is not None:
        anchors = anchors[:MAX_ANCHORS]

    Path(DEBUG_DIR).mkdir(parents=True, exist_ok=True)

    all_frame_counts = []
    window_counts = []
    window_stable_counts_debug = []
    max_count_frames = []

    best_count = 0
    debug_saved = 0
    total_yolo_frames = 0

    for anchor_i, anchor in enumerate(anchors):
        start_t = max(0, anchor - WINDOW_SIZE / 2)
        end_t = min(duration, anchor + WINDOW_SIZE / 2)

        start_frame = int(start_t * fps)
        end_frame = int(end_t * fps)

        start_frame = max(0, min(start_frame, total_frames - 1))
        end_frame = max(0, min(end_frame, total_frames - 1))

        frame_counts_this_window = []

        for frame_idx in range(start_frame, end_frame + 1, FRAME_STRIDE):
            cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
            ret, frame = cap.read()

            if not ret:
                continue

            t = frame_idx / fps

            dog_boxes = detect_dog_boxes(frame, model)
            dog_count = len(dog_boxes)

            all_frame_counts.append(dog_count)
            frame_counts_this_window.append(dog_count)
            total_yolo_frames += 1

            record = {
                "anchor_index": anchor_i,
                "anchor_time": round(float(anchor), 3),
                "frame_idx": int(frame_idx),
                "time": round(float(t), 3),
                "dog_count": int(dog_count),
                "dog_boxes": dog_boxes
            }

            if dog_count > best_count:
                best_count = dog_count
                max_count_frames = [record]

                if SAVE_DEBUG_IMAGES and debug_saved < MAX_DEBUG_IMAGES_PER_VIDEO:
                    drawn = draw_dog_boxes(frame, dog_boxes, video_id, frame_idx, t)

                    debug_path = os.path.join(
                        DEBUG_DIR,
                        f"video_{video_id}_count_{dog_count}_frame_{frame_idx}_time_{t:.2f}.jpg"
                    )

                    cv2.imwrite(debug_path, drawn)
                    record["debug_image"] = debug_path
                    debug_saved += 1

            elif dog_count == best_count and dog_count > 0:
                max_count_frames.append(record)

        window_count = max(frame_counts_this_window) if frame_counts_this_window else 0
        window_stable = stable_count_debug(frame_counts_this_window)

        window_counts.append(int(window_count))
        window_stable_counts_debug.append(int(window_stable))

    cap.release()

    final_count = max(all_frame_counts) if all_frame_counts else 0

    info = {
        "error": "",
        "duration": float(duration),
        "fps": float(fps),
        "total_frames": int(total_frames),
        "video_type": video_type,
        "num_anchors": len(anchors),
        "anchor_times": [round(float(a), 3) for a in anchors],
        "window_size": float(WINDOW_SIZE),
        "frame_stride": int(FRAME_STRIDE),
        "window_counts": window_counts,
        "window_stable_counts_debug": window_stable_counts_debug,
        "all_frame_counts_summary": {
            "max": int(max(all_frame_counts)) if all_frame_counts else 0,
            "min": int(min(all_frame_counts)) if all_frame_counts else 0,
            "num_frames": len(all_frame_counts),
            "counts_histogram": {
                str(k): int(v)
                for k, v in Counter(all_frame_counts).items()
            }
        },
        "total_yolo_frames": int(total_yolo_frames),
        "max_count_frames": max_count_frames[:5],
        "final_count": int(final_count)
    }

    return int(final_count), info

In [ ]:
# =========================
# VALIDATION CONFIG
# =========================

# Reuse METADATA_PATH from the top CONFIG cell.
metadata_path = METADATA_PATH

GROUND_TRUTH_COUNTS = {
    3766: 1,
    5505: 1,
    196669: 1,
    196810: 1,
    134: 5,
    151: 1,
    260: 2,
    811: 1,
    3776: 1,
    3871: 1,
    4596: 2,

    24: 2,
    28: 2,
    29: 1,
    77: 1,
    86: 2,
    123: 2,
    146: 2,
    152: 2,
    172: 1,
    176: 1,
    190: 1,
    195: 2,
    198: 1,
    245: 1,
    246: 1,
    269: 1,
    322: 1,
    330: 2,
    334: 2,
    422: 2,
    411: 1,

    253: 2,
    255: 2,
    261: 1,
    262: 3,
    271: 4,
    275: 1,
    286: 3,
    321: 1,
    366: 8,
    417: 1,
    418: 1,
    419: 3,

    2151: 3,
    1716: 2,
    1673: 1,
    1552: 2,
    1458: 2,
    1445: 2,
    1399: 1,
    1356: 2,
    1328: 1,
    1255: 1,
    1230: 3,
    1229: 3,
    944: 3,
    723: 3,
    722: 3,
    485: 4,
    478: 4,
    472: 3,
    453: 3,
    442: 3,
    439: 3,
}

TEST_VIDEO_IDS = set(GROUND_TRUTH_COUNTS.keys())

# Use same anchor/window plan as old notebook
COVERAGE = 0.40
WINDOW_SIZE = 2.0
MAX_ANCHORS = None

# New notebook setting: continuous frames inside each anchor window
FRAME_STRIDE = 3

# Test only these 4 thresholds first
THRESHOLDS = [0.20, 0.23, 0.26, 0.29]

# Keep YOLO NMS fixed
YOLO_INTERNAL_CONF = 0.05
YOLO_NMS_IOU = 0.45
IMGSZ = 1280
DEVICE = 0

SAVE_DEBUG_IMAGES = False   # turn off for validation so it runs faster

In [12]:
# =========================
# LOAD TRUTH-LABELED VIDEOS
# =========================

with open(metadata_path, "r") as f:
    all_videos = json.load(f)

videos = [
    v for v in all_videos
    if int(v["video_id"]) in TEST_VIDEO_IDS
]

videos = sorted(videos, key=lambda v: int(v["video_id"]))

print(f"Found {len(videos)} truth-labeled videos")
print([int(v["video_id"]) for v in videos])

Found 65 truth-labeled videos
[24, 28, 29, 77, 86, 123, 134, 146, 151, 152, 172, 176, 190, 195, 198, 245, 246, 253, 255, 260, 261, 262, 269, 271, 275, 286, 321, 322, 330, 334, 366, 411, 417, 418, 419, 422, 439, 442, 453, 472, 478, 485, 722, 723, 811, 944, 1229, 1230, 1255, 1328, 1356, 1399, 1445, 1458, 1552, 1673, 1716, 2151, 3766, 3776, 3871, 4596, 5505, 196669, 196810]


run the conf of .32-.34 and the truth table

In [ ]:
# =========================
# RUN NEW CONTINUOUS-WINDOW TRUTH VALIDATION
# CONF 0.32 TO 0.34
# =========================

import os
import json
import time
import pandas as pd

# Make sure these already exist from earlier cells:
# model
# videos
# GROUND_TRUTH_COUNTS
# estimate_num_dogs_continuous_windows
# fix_video_path
# OUTPUT_DIR (from top CONFIG cell)

THRESHOLDS = [0.320, 0.325, 0.330, 0.335, 0.340]

# Continuous-window settings
COVERAGE = 0.40
WINDOW_SIZE = 2.0
MAX_ANCHORS = None
FRAME_STRIDE = 3

# YOLO settings
YOLO_INTERNAL_CONF = 0.05
YOLO_NMS_IOU = 0.45
IMGSZ = 1280
DEVICE = 0

# Turn off debug images for validation speed
SAVE_DEBUG_IMAGES = False

all_threshold_summaries = []

for threshold in THRESHOLDS:
    print("\n==============================")
    print(f"NEW continuous-window validation, CONF_THRESHOLD = {threshold}")
    print("==============================")

    # detect_dog_boxes() uses global CONF_THRESHOLD
    CONF_THRESHOLD = threshold

    conf_name = str(threshold).replace(".", "p")
    output_json = f"{OUTPUT_DIR}/yolo26_NEW_continuous_validation_conf{conf_name}.json"

    results = []

    for i, video in enumerate(videos):
        video_id = int(video["video_id"])
        video_identifier = video.get("video_identifier", "")
        video_path = fix_video_path(video["video_path"])

        try:
            duration = float(video["duration"])
        except Exception:
            duration = None

        manual_count = int(GROUND_TRUTH_COUNTS[video_id])

        print(f"[{i+1}/{len(videos)}] video_id={video_id}")

        start_time = time.time()

        try:
            count, info = estimate_num_dogs_continuous_windows(
                video_path=video_path,
                video_id=video_id,
                duration=duration
            )

            error = info.get("error", "")

        except Exception as e:
            count = -1
            info = {}
            error = str(e)

        runtime_seconds = time.time() - start_time

        result = {
            "method": "new_continuous_max",
            "video_id": video_id,
            "video_identifier": video_identifier,
            "video_path": video_path,
            "duration": duration,

            "manual_count": manual_count,
            "num_dogs_visible_estimated": int(count),

            "correct": int(count) == manual_count if count >= 0 else False,
            "absolute_error": abs(int(count) - manual_count) if count >= 0 else None,

            "conf_threshold": threshold,
            "yolo_internal_conf": YOLO_INTERNAL_CONF,
            "yolo_nms_iou": YOLO_NMS_IOU,
            "imgsz": IMGSZ,
            "coverage": COVERAGE,
            "window_size": WINDOW_SIZE,
            "frame_stride": FRAME_STRIDE,

            "video_type": info.get("video_type", ""),
            "num_anchors": info.get("num_anchors", None),
            "total_yolo_frames": info.get("total_yolo_frames", None),
            "runtime_seconds": round(runtime_seconds, 2),
            "error": error,
            "debug_info": info,
        }

        results.append(result)

        # save after every video so progress is not lost
        with open(output_json, "w") as f:
            json.dump(results, f, indent=2)

    df = pd.DataFrame(results)
    valid = df[df["error"].fillna("") == ""].copy()

    summary = {
        "method": "new_continuous_max",
        "conf_threshold": threshold,
        "correct_count": int(valid["correct"].sum()),
        "total_videos": len(valid),
        "accuracy": valid["correct"].mean(),
        "mean_absolute_error": valid["absolute_error"].mean(),
        "output_json": output_json,
    }

    all_threshold_summaries.append(summary)

    print("\nSummary for threshold:", threshold)
    print(summary)

summary_df = pd.DataFrame(all_threshold_summaries)

print("\nFinal NEW continuous-window threshold comparison:")
display(summary_df)

best = summary_df.sort_values(
    by=["accuracy", "mean_absolute_error"],
    ascending=[False, True]
).iloc[0]

print("\nBest NEW continuous-window threshold:")
print(best)

compare old thruth from .32 to .34 to new

In [ ]:
# =========================
# COMPARE OLD VS NEW FOR 0.32 TO 0.34
# =========================

import os
import json
import pandas as pd

THRESHOLDS = [0.320, 0.325, 0.330, 0.335, 0.340]

comparison_rows = []

for threshold in THRESHOLDS:
    conf_name = str(threshold).replace(".", "p")

    paths = {
        "old_sparse_stable": f"{OUTPUT_DIR}/yolo26_validation_conf{conf_name}.json",
        "new_continuous_max": f"{OUTPUT_DIR}/yolo26_NEW_continuous_validation_conf{conf_name}.json",
    }

    for method, path in paths.items():
        if not os.path.exists(path):
            print("Missing:", path)
            continue

        with open(path, "r") as f:
            results = json.load(f)

        df = pd.DataFrame(results)

        if "error" in df.columns:
            valid = df[df["error"].fillna("") == ""].copy()
        else:
            valid = df.copy()

        comparison_rows.append({
            "method": method,
            "conf_threshold": threshold,
            "correct_count": int(valid["correct"].sum()),
            "total_videos": len(valid),
            "accuracy": valid["correct"].mean(),
            "mean_absolute_error": valid["absolute_error"].mean(),
            "output_json": path,
        })

comparison_df = pd.DataFrame(comparison_rows)

comparison_df = comparison_df.sort_values(
    by=["method", "conf_threshold"]
)

display(comparison_df)

best_overall = comparison_df.sort_values(
    by=["accuracy", "mean_absolute_error"],
    ascending=[False, True]
).iloc[0]

print("\nBest overall:")
print(best_overall)

run the code

In [ ]:
# =========================
# RUN NEW CONTINUOUS-WINDOW VALIDATION
# =========================

import pandas as pd
import time
import json

all_new_threshold_summaries = []

for threshold in THRESHOLDS:
    print("\n==============================")
    print(f"NEW continuous-window validation, CONF_THRESHOLD = {threshold}")
    print("==============================")

    # IMPORTANT:
    # Your detect_dog_boxes() function uses global CONF_THRESHOLD,
    # so update it before each threshold run.
    CONF_THRESHOLD = threshold

    output_json = f"{OUTPUT_DIR}/yolo26_NEW_continuous_validation_conf{int(threshold * 100):03d}.json"

    results = []

    for i, video in enumerate(videos):
        video_id = int(video["video_id"])
        video_identifier = video.get("video_identifier", "")
        video_path = fix_video_path(video["video_path"])
        duration = float(video["duration"])
        manual_count = GROUND_TRUTH_COUNTS[video_id]

        print(f"[{i+1}/{len(videos)}] video_id={video_id}")

        start_time = time.time()

        try:
            count, info = estimate_num_dogs_continuous_windows(
                video_path=video_path,
                video_id=video_id,
                duration=duration
            )

            error = info.get("error", "")

        except Exception as e:
            count = -1
            info = {}
            error = str(e)

        runtime_seconds = time.time() - start_time
        total_yolo_frames = info.get("total_yolo_frames", 0)

        result = {
            "method": "new_continuous_max",
            "video_id": video_id,
            "video_identifier": video_identifier,
            "video_path": video_path,
            "duration": duration,

            "manual_count": manual_count,
            "num_dogs_visible_estimated": count,
            "correct": count == manual_count if count >= 0 else False,
            "absolute_error": abs(count - manual_count) if count >= 0 else None,

            "runtime_seconds": runtime_seconds,
            "total_yolo_frames": total_yolo_frames,
            "effective_fps": total_yolo_frames / runtime_seconds if runtime_seconds > 0 else 0,

            "fps": info.get("fps", None),
            "video_type": info.get("video_type", None),
            "num_anchors": info.get("num_anchors", None),
            "anchor_times": info.get("anchor_times", []),
            "window_counts": info.get("window_counts", []),

            "coverage": COVERAGE,
            "window_size": WINDOW_SIZE,
            "frame_stride": FRAME_STRIDE,
            "conf_threshold": threshold,

            "error": error
        }

        results.append(result)

        # Save after every video so progress is not lost
        with open(output_json, "w") as f:
            json.dump(results, f, indent=2)

    df = pd.DataFrame(results)
    valid = df[df["error"].fillna("") == ""]

    accuracy = valid["correct"].mean()
    mean_abs_error = valid["absolute_error"].mean()
    correct_count = int(valid["correct"].sum())

    all_new_threshold_summaries.append({
        "method": "new_continuous_max",
        "conf_threshold": threshold,
        "correct_count": correct_count,
        "total_videos": len(valid),
        "accuracy": accuracy,
        "mean_absolute_error": mean_abs_error,
        "output_json": output_json
    })

new_summary_df = pd.DataFrame(all_new_threshold_summaries)

print("\nNEW continuous-window threshold comparison:")
print(new_summary_df)

best_new = new_summary_df.sort_values(
    by=["accuracy", "mean_absolute_error"],
    ascending=[False, True]
).iloc[0]

print("\nBest NEW threshold:")
print(best_new)

In [ ]:
# =========================
# COMPARE OLD VS NEW
# =========================

import os
import json
import pandas as pd

comparison_rows = []

# Old sparse/stable results
for t in [20, 23, 26, 29]:
    threshold = t / 100
    path = f"{OUTPUT_DIR}/yolo26_validation_conf{t:03d}.json"

    if not os.path.exists(path):
        print("Missing old result:", path)
        continue

    with open(path, "r") as f:
        results = json.load(f)

    df = pd.DataFrame(results)
    valid = df[df["error"].fillna("") == ""]

    comparison_rows.append({
        "method": "old_sparse_stable",
        "conf_threshold": threshold,
        "correct_count": int(valid["correct"].sum()),
        "total_videos": len(valid),
        "accuracy": valid["correct"].mean(),
        "mean_absolute_error": valid["absolute_error"].mean()
    })

# New continuous/max results
for t in [20, 23, 26, 29]:
    threshold = t / 100
    path = f"{OUTPUT_DIR}/yolo26_NEW_continuous_validation_conf{t:03d}.json"

    if not os.path.exists(path):
        print("Missing new result:", path)
        continue

    with open(path, "r") as f:
        results = json.load(f)

    df = pd.DataFrame(results)
    valid = df[df["error"].fillna("") == ""]

    comparison_rows.append({
        "method": "new_continuous_max",
        "conf_threshold": threshold,
        "correct_count": int(valid["correct"].sum()),
        "total_videos": len(valid),
        "accuracy": valid["correct"].mean(),
        "mean_absolute_error": valid["absolute_error"].mean()
    })

comparison_df = pd.DataFrame(comparison_rows)

comparison_df = comparison_df.sort_values(
    by=["method", "conf_threshold"]
)

print(comparison_df)

best_overall = comparison_df.sort_values(
    by=["accuracy", "mean_absolute_error"],
    ascending=[False, True]
).iloc[0]

print("\nBest overall method/threshold:")
print(best_overall)

In [ ]:
# =========================
# WRONG VIDEOS FOR NEW METHOD
# =========================

threshold_to_check = 0.23
t = int(threshold_to_check * 100)

path = f"{OUTPUT_DIR}/yolo26_NEW_continuous_validation_conf{t:03d}.json"

with open(path, "r") as f:
    results = json.load(f)

df = pd.DataFrame(results)
valid = df[df["error"].fillna("") == ""]

wrong = valid[valid["correct"] == False].copy()

print(f"New method wrong videos at threshold {threshold_to_check}:")
print(wrong[
    [
        "video_id",
        "manual_count",
        "num_dogs_visible_estimated",
        "absolute_error",
        "video_type",
        "num_anchors",
        "total_yolo_frames"
    ]
].to_string(index=False))

In [22]:
# =========================
# LOAD METADATA AND SELECT VIDEOS
# =========================

with open(METADATA_PATH, "r") as f:
    all_videos = json.load(f)

wanted_ids = set(FAILED_ZERO_VIDEO_IDS)

videos = []

for v in all_videos:
    try:
        video_id = int(v["video_id"])
    except Exception:
        continue

    if wanted_ids is not None and video_id not in wanted_ids:
        continue

    raw_duration = v.get("duration", None)

    if raw_duration is None:
        continue

    try:
        duration = float(raw_duration)
    except Exception:
        continue

    if duration <= 0:
        continue

    if MAX_DURATION is not None and duration > MAX_DURATION:
        continue

    videos.append(v)

videos = sorted(videos, key=lambda x: int(x["video_id"]))

if MAX_VIDEOS is not None:
    videos = videos[:MAX_VIDEOS]

print(f"Selected {len(videos)} videos")
print([v["video_id"] for v in videos[:10]])

Selected 5 videos
[2, 3, 6, 8, 130]


In [23]:
# =========================
# RUN BATCH
# =========================

results = []

for i, video in enumerate(videos):
    video_id = int(video["video_id"])
    video_identifier = video.get("video_identifier", "")
    video_path = fix_video_path(video.get("video_path", ""))
    duration = float(video["duration"])

    print("\n==============================")
    print(f"[{i + 1}/{len(videos)}] video_id={video_id}")
    print(f"duration={duration:.2f}")
    print(f"path={video_path}")
    print("==============================")

    start_time = time.time()

    try:
        count, info = estimate_num_dogs_continuous_windows(
            video_path=video_path,
            video_id=video_id,
            duration=duration
        )

        error = info.get("error", "")

    except Exception as e:
        count = -1
        info = {}
        error = str(e)

    runtime_seconds = time.time() - start_time

    result = {
        "video_id": video_id,
        "video_identifier": video_identifier,
        "video_path": video_path,
        "duration": duration,
        "num_dogs_visible_estimated": int(count),
        "conf_threshold": CONF_THRESHOLD,
        "yolo_internal_conf": YOLO_INTERNAL_CONF,
        "yolo_nms_iou": YOLO_NMS_IOU,
        "imgsz": IMGSZ,
        "window_size": WINDOW_SIZE,
        "frame_stride": FRAME_STRIDE,
        "runtime_seconds": round(runtime_seconds, 2),
        "error": error,
        "debug_info": info
    }

    results.append(result)

    print(f"Estimated dogs: {count}")
    print(f"Runtime: {runtime_seconds:.2f}s")

    if error:
        print(f"ERROR: {error}")

    # Save after every video so progress is not lost.
    with open(OUTPUT_JSON, "w") as f:
        json.dump(results, f, indent=2)

print("\nDone.")
print(f"Saved {len(results)} results to {OUTPUT_JSON}")


[1/5] video_id=2
duration=8.00
path=/redgiant/data/shared/private/dogs/raw_videos/dog_videos/chihuahua/output/Li Li/dimBAhwoUsg.mp4
Estimated dogs: 0
Runtime: 1.28s

[2/5] video_id=3
duration=27.00
path=/redgiant/data/shared/private/dogs/raw_videos/dog_videos/chihuahua/output/ゆりぴ〜チャンネル/Bp5QThpDDXM.mp4
Estimated dogs: 1
Runtime: 1.39s

[3/5] video_id=6
duration=37.00
path=/redgiant/data/shared/private/dogs/raw_videos/shiba/output/lb/Franz Biedermann/B6UDdBGTozM.mp4
Estimated dogs: 0
Runtime: 5.72s

[4/5] video_id=8
duration=12.00
path=/redgiant/data/shared/private/dogs/raw_videos/shiba/output/lb/ScatzerZ 200M Views/BfJauwaqyF0.mp4
Estimated dogs: 1
Runtime: 1.53s

[5/5] video_id=130
duration=44.00
path=/redgiant/data/shared/private/dogs/raw_videos/dog_videos/chihuahua/output/Theo Evans/4cAAGk9cW60.mp4
Estimated dogs: 0
Runtime: 5.17s

Done.
Saved 5 results to /supernova/data/home/aryah/yolo26_continuous_failed_zero_retest_conf025.json


In [24]:
recovered = []
still_zero = []
errors = []

for r in results:
    vid = r["video_id"]
    pred = r["num_dogs_visible_estimated"]

    if r["error"]:
        errors.append(vid)
    elif pred == 0:
        still_zero.append(vid)
    else:
        recovered.append((vid, pred))

print("Recovered from 0:")
print(recovered)

print("\nStill 0:")
print(still_zero)

print("\nErrors:")
print(errors)

print("\nSummary:")
print("Recovered:", len(recovered))
print("Still zero:", len(still_zero))
print("Errors:", len(errors))

Recovered from 0:
[(3, 1), (8, 1)]

Still 0:
[2, 6, 130]

Errors:
[]

Summary:
Recovered: 2
Still zero: 3
Errors: 0


In [25]:
for r in results:
    vid = r["video_id"]
    pred = r["num_dogs_visible_estimated"]

    print("\n======================")
    print("video_id:", vid)
    print("new estimate:", pred)

    frames = r["debug_info"].get("max_count_frames", [])

    for f in frames[:3]:
        print("time:", f.get("time"), "count:", f.get("dog_count"))

        if "debug_image" in f:
            print("debug image:", f["debug_image"])


video_id: 2
new estimate: 0

video_id: 3
new estimate: 1
time: 2.3 count: 1
debug image: /supernova/data/home/aryah/yolo26_debug_failed_zero_retest/video_3_count_1_frame_69_time_2.30.jpg
time: 2.5 count: 1
time: 2.6 count: 1

video_id: 6
new estimate: 0

video_id: 8
new estimate: 1
time: 2.461 count: 1
debug image: /supernova/data/home/aryah/yolo26_debug_failed_zero_retest/video_8_count_1_frame_59_time_2.46.jpg
time: 5.088 count: 1
time: 9.218 count: 1

video_id: 130
new estimate: 0


In [ ]:
# =========================
# QUICK RESULT CHECK
# =========================

counts = Counter(
    r["num_dogs_visible_estimated"]
    for r in results
    if r["num_dogs_visible_estimated"] >= 0
)

counts

In [ ]:
# =========================
# PRINT RESULT ROWS
# =========================

for r in results:
    print(
        r["video_id"],
        "estimated:",
        r["num_dogs_visible_estimated"],
        "error:",
        r["error"]
    )

In [ ]:
# =========================
# SHOW DEBUG IMAGE PATHS
# =========================

for r in results:
    print("\nvideo_id:", r["video_id"])
    frames = r["debug_info"].get("max_count_frames", [])

    for f in frames[:3]:
        if "debug_image" in f:
            print(f["debug_image"])

## What to do after this works

1. Run `MAX_VIDEOS = 5`.
2. Inspect the debug images in `DEBUG_DIR`.
3. If boxes look reasonable, run 20 manually labeled videos.
4. Then run all 64 manually labeled videos.
5. Only after validation, run 1,000 videos.

Do not add tracking yet. This notebook is testing one clean idea:

**continuous-window YOLO counting using max visible dog boxes.**

## Step 5 — Re-aggregation Diagnostic (No YOLO Inference)

Goal: figure out whether the gap between `continuous_max` (~20% accuracy) and
`old_sparse_stable` (~52% accuracy) comes from the **aggregation rule** (`max`
vs `stable_count`) or from the **frame extraction loop** itself.

This step is cheap — it reads the existing
`yolo26_NEW_continuous_validation_conf<conf>p<rest>.json` files produced by the
validation cell above (which already store `debug_info.all_frame_counts_summary.counts_histogram`
per video), reconstructs the per-frame count list, and re-applies the
`stable_count` rule lifted directly from the sparse notebook.

If `stable_count` on the same data closes the accuracy gap, the bug is the
aggregation. If it doesn't, the bug is upstream (frame extraction / detection).

No YOLO inference happens here — safe to run anywhere with the existing JSONs.

In [ ]:
# =========================
# STEP 5 - HELPERS
# =========================

import os
import json
import math
from collections import Counter

import pandas as pd


def stable_count(counts, min_fraction=0.30, min_frames=3):
    """
    Most-frequent count if it appears in at least max(min_frames, 30%) of frames.
    Lifted unchanged from copy-dog-bounding-box.ipynb (sparse notebook), cell 6.
    """

    if not counts:
        return 0

    threshold = max(min_frames, math.ceil(min_fraction * len(counts)))
    freq = Counter(counts)

    for count in sorted(freq.keys(), reverse=True):
        if freq[count] >= threshold:
            return count

    return 0


def reconstruct_counts_from_histogram(hist):
    """
    The continuous estimator stores Counter(all_frame_counts) as a dict with
    string keys in debug_info.all_frame_counts_summary.counts_histogram.
    Expand back to the flat list so stable_count() can be applied.
    """

    counts = []

    if not hist:
        return counts

    for k, v in hist.items():
        counts.extend([int(k)] * int(v))

    return counts


def step5_load_continuous_results(threshold, output_dir=None):
    """
    Load the continuous-window validation JSON for one conf threshold.
    The file is the one written by the 'CONF 0.32 TO 0.34' validation cell,
    which uses the 'p' separator naming and saves debug_info per video.
    """

    if output_dir is None:
        output_dir = OUTPUT_DIR

    conf_name = str(threshold).replace(".", "p")
    path = f"{output_dir}/yolo26_NEW_continuous_validation_conf{conf_name}.json"

    if not os.path.exists(path):
        return path, None

    with open(path, "r") as f:
        return path, json.load(f)


def step5_reaggregate(record):
    """
    Compute alternate aggregations from one per-video record. Returns:
      continuous_max                       : the original max-based prediction
      continuous_stable_global             : stable_count over all sampled frames
      continuous_window_stable_then_max    : max of per-window stable counts (already in debug_info)
      num_frames_processed                 : how many frames actually got a YOLO call
    """

    info = record.get("debug_info", {}) or {}
    summary = info.get("all_frame_counts_summary", {}) or {}
    hist = summary.get("counts_histogram")

    num_frames = info.get("total_yolo_frames")
    if num_frames is None:
        num_frames = summary.get("num_frames")

    out = {
        "continuous_max": record.get("num_dogs_visible_estimated"),
        "continuous_stable_global": None,
        "continuous_window_stable_then_max": None,
        "num_frames_processed": num_frames,
    }

    if hist:
        counts = reconstruct_counts_from_histogram(hist)
        out["continuous_stable_global"] = stable_count(counts)

    wsc = info.get("window_stable_counts_debug")
    if wsc:
        out["continuous_window_stable_then_max"] = int(max(wsc))

    return out

In [ ]:
# =========================
# STEP 5 - RE-AGGREGATION COMPARISON
# =========================

# Confidence thresholds whose continuous output JSONs we want to re-aggregate.
# These match the thresholds the 'CONF 0.32 TO 0.34' validation cell wrote.
STEP5_THRESHOLDS = [0.320, 0.325, 0.330, 0.335, 0.340]

per_video_rows = []

for threshold in STEP5_THRESHOLDS:
    path, records = step5_load_continuous_results(threshold)

    if records is None:
        print(f"Missing: {path}")
        continue

    print(f"Loaded {len(records)} records from {path}")

    n_with_hist = 0

    for r in records:
        # Drop videos with errors so they do not contaminate the comparison.
        if r.get("error"):
            continue

        agg = step5_reaggregate(r)

        if agg["continuous_stable_global"] is not None:
            n_with_hist += 1

        per_video_rows.append({
            "conf_threshold": threshold,
            "video_id": r.get("video_id"),
            "ground_truth": r.get("manual_count"),
            **agg,
        })

    print(f"  videos with usable counts_histogram: {n_with_hist}")

per_video_df = pd.DataFrame(per_video_rows)

# Summarize accuracy / MAE per aggregation per threshold.
summary_rows = []

for threshold in STEP5_THRESHOLDS:
    df_t = per_video_df[per_video_df["conf_threshold"] == threshold]
    df_t = df_t.dropna(subset=["continuous_stable_global"])

    if len(df_t) == 0:
        continue

    for col in [
        "continuous_max",
        "continuous_stable_global",
        "continuous_window_stable_then_max",
    ]:
        col_series = df_t[col].dropna()

        if len(col_series) == 0:
            continue

        # Align ground_truth to the rows that have this column populated.
        gt = df_t.loc[col_series.index, "ground_truth"]
        correct = int((col_series == gt).sum())
        mae = (col_series - gt).abs().mean()

        summary_rows.append({
            "conf_threshold": threshold,
            "aggregation": col,
            "correct_count": correct,
            "total_videos": len(col_series),
            "accuracy": correct / len(col_series),
            "mean_absolute_error": mae,
        })

summary_df = pd.DataFrame(summary_rows)

if len(summary_df) > 0:
    summary_df = summary_df.sort_values(by=["conf_threshold", "aggregation"])

    print("\nStep 5 - aggregation comparison (continuous_max vs continuous_stable_reaggregated vs ground truth):")
    display(summary_df)

    # Pick the best (accuracy first, then lowest MAE) for a quick recommendation.
    best = summary_df.sort_values(
        by=["accuracy", "mean_absolute_error"],
        ascending=[False, True],
    ).iloc[0]

    print("\nBest aggregation overall (from the existing continuous JSONs):")
    print(best)
else:
    print("\nNo Step 5 results - check that the conf*.json files exist with debug_info.")

In [ ]:
# =========================
# STEP 5 - PER-VIDEO BUCKET ANALYSIS
# =========================

# Pick one threshold to drill into. 0.330 is a reasonable middle.
DIAG_THRESHOLD = 0.330

diag = per_video_df[per_video_df["conf_threshold"] == DIAG_THRESHOLD].copy()
diag = diag.dropna(subset=["continuous_stable_global"]).reset_index(drop=True)

if len(diag) == 0:
    print(f"No usable rows at conf={DIAG_THRESHOLD}. Run the validation cell or pick another threshold.")
else:
    diag["max_correct"] = diag["continuous_max"] == diag["ground_truth"]
    diag["stable_correct"] = diag["continuous_stable_global"] == diag["ground_truth"]

    bucket_a = diag[(diag["continuous_max"] == 0) & (diag["ground_truth"] > 0)]
    bucket_b = diag[diag["continuous_max"] > diag["ground_truth"] + 1]
    recovered = diag[(~diag["max_correct"]) & (diag["stable_correct"])]
    regressed = diag[(diag["max_correct"]) & (~diag["stable_correct"])]

    print(f"At conf={DIAG_THRESHOLD} ({len(diag)} videos with usable histograms):")
    print(f"  Bucket A (continuous_max=0 but ground_truth>0): {len(bucket_a)} videos")
    print(f"  Bucket B (continuous_max > ground_truth + 1):   {len(bucket_b)} videos")
    print(f"  Recovered by switching max -> stable_count:     {len(recovered)} videos")
    print(f"  Regressed by switching max -> stable_count:     {len(regressed)} videos")

    cols = [
        "video_id",
        "ground_truth",
        "continuous_max",
        "continuous_stable_global",
        "continuous_window_stable_then_max",
        "num_frames_processed",
    ]

    if len(bucket_a) > 0:
        print("\nBucket A - 'impossible' undercounts (max=0 but truth>0):")
        print(bucket_a[cols].to_string(index=False))

    if len(bucket_b) > 0:
        print("\nBucket B - max overcounts by more than 1:")
        print(bucket_b[cols].to_string(index=False))

    if len(recovered) > 0:
        print("\nVideos recovered by stable_count:")
        print(recovered[cols].to_string(index=False))

    if len(regressed) > 0:
        print("\nVideos that get worse under stable_count (sanity check):")
        print(regressed[cols].to_string(index=False))